# SICD Orthorectification Demo — Slant Range vs. Terrain-Corrected

**Source script:** `grdl/example/ortho/sicd_ortho_demo.py`

This notebook demonstrates the **visual impact of orthorectification** on SAR imagery:

1. **Slant-range geometry** — native SAR collection plane (row=azimuth, col=slant range)
2. **Orthorectified geometry** — WGS-84 geographic grid with DEM terrain correction

## Why This Matters

SAR images are collected in the **slant plane** — a tilted coordinate system defined by the radar line-of-sight. For a scene with graze angle ~45°, the slant-range geometry is **visibly stretched** compared to true geographic coordinates:

- **Slant range** encodes distance from radar → objects at higher elevation appear **closer** to the sensor
- **Azimuth** encodes along-track position → roughly aligned with north, but distorted by platform motion

**Orthorectification** transforms SAR imagery into a **geographically correct** representation:
- Removes range-dependent compression
- Corrects for terrain elevation (layover, foreshortening)
- Enables overlay with maps, vector data, and other imagery

## Expected Visual Difference

For a scene with graze angle ~45° (Savannah, Georgia example):
- **Slant range image** → vertically compressed (ground range is shorter than slant range)
- **Orthorectified image** → geometrically correct, visibly **taller** (ground range restored)

## Data Setup

Set `SICD_PATH` and `DEM_PATH` in Cell 3 to point to:
- A SICD NITF file (complex SAR image)
- A DEM directory (FABDEM, DTED, or GeoTIFF tiles)

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np

from grdl.IO import get_reader
from grdl.data_prep import ChipExtractor
from grdl.geolocation import ChipGeolocation, SICDGeolocation
from grdl.geolocation.elevation import open_elevation
from grdl.image_processing.ortho import orthorectify

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Set your SICD and DEM paths here
# ════════════════════════════════════════════════════════════════════
SICD_PATH = Path("/path/to/your/sicd_file.nitf")  # ← CHANGE THIS to your SICD file
DEM_PATH = Path("/path/to/dem/directory")          # ← CHANGE THIS to DEM directory (FABDEM, DTED, etc.)
CHIP_SIZE = 2048 * 4  # pixels per side (8192 = large chip for clear visual comparison)
INTERPOLATION = 'bilinear'  # ortho interpolation method

assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"
assert DEM_PATH.exists(), f"DEM path not found: {DEM_PATH}"
print(f"Target: {SICD_PATH.name}")
print(f"DEM: {DEM_PATH}")

## 1. Open SICD & Inspect Metadata

Extract key metadata:
- **SCP** (Scene Center Point) — geographic reference for the image
- **Graze angle** — radar depression angle at SCP (0° = horizontal, 90° = nadir)
- **Grid parameters** — row/col spacing, image extent

**Graze angle interpretation:**
- Low graze (< 30°) → extreme layover, severe foreshortening
- Medium graze (30-60°) → moderate geometric distortion ← **typical for airborne SAR**
- High graze (> 60°) → minimal layover, nearly orthographic

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = meta.rows, meta.cols
    scp = meta.geo_data.scp.llh
    graze = meta.scpcoa.graze_ang
    
    print(f"Full image size : {rows} x {cols}")
    print(f"SCP location    : ({scp.lat:.4f}°, {scp.lon:.4f}°)")
    print(f"Graze angle     : {graze:.2f}°")
    print(f"\nGraze interpretation: ", end="")
    if graze < 30:
        print("LOW — severe layover expected")
    elif graze < 60:
        print("MEDIUM — moderate geometric distortion (typical airborne)")
    else:
        print("HIGH — minimal layover, nearly orthographic")
    
    # Plan center chip
    extractor = ChipExtractor(nrows=rows, ncols=cols)
    region = extractor.chip_at_point(
        rows // 2, cols // 2,
        row_width=CHIP_SIZE, col_width=CHIP_SIZE,
    )
    chip_rows = region.row_end - region.row_start
    chip_cols = region.col_end - region.col_start
    print(f"\nChip dimensions : {chip_rows} x {chip_cols}")
    print(f"Chip location   : row [{region.row_start}:{region.row_end}], col [{region.col_start}:{region.col_end}]")
    
    # Build geolocation for full image, then wrap for chip
    geo_full = SICDGeolocation.from_reader(reader)
    geo_chip = ChipGeolocation(
        geo_full,
        row_offset=region.row_start,
        col_offset=region.col_start,
        shape=(chip_rows, chip_cols),
    )
    
    # Attach DEM for terrain-corrected R/Rdot projection
    geo_chip.elevation = open_elevation(
        DEM_PATH,
        location=(float(scp.lat), float(scp.lon)),
    )
    print(f"\nDEM model       : {type(geo_chip.elevation).__name__}")
    
    # Read complex chip
    complex_chip = reader.read_chip(
        region.row_start, region.row_end,
        region.col_start, region.col_end,
    )

print(f"\nComplex chip shape: {complex_chip.shape}, dtype: {complex_chip.dtype}")
print(f"Memory: {complex_chip.nbytes / 1e6:.1f} MB")

## 2. Magnitude Extraction

**Critical:** Convert complex → magnitude **before** orthorectification.

Why? Resampling complex samples causes **phase cancellation** — neighboring complex values with different phases destructively interfere during interpolation. Always extract magnitude (or power) first, then resample the real-valued result.

In [ ]:
magnitude = np.abs(complex_chip).astype(np.float32)
del complex_chip  # free memory

print(f"Magnitude shape: {magnitude.shape}, dtype: {magnitude.dtype}")
print(f"Range: [{magnitude.min():.2e}, {magnitude.max():.2e}]")
print(f"Mean: {magnitude.mean():.2e}, Std: {magnitude.std():.2e}")

## 3. Orthorectification

Transform the slant-plane magnitude image onto a WGS-84 geographic grid with DEM terrain correction.

**Algorithm:**
1. Auto-compute output grid bounds from SICD metadata (lat/lon extent)
2. Auto-compute pixel spacing to match SICD ground resolution
3. For each output pixel:
   - Query DEM for terrain height
   - Inverse project (lat, lon, height) → (row, col) via R/Rdot iteration
   - Interpolate magnitude at fractional (row, col)

**Output:**
- `result.data` — orthorectified magnitude array
- `result.output_grid` — geographic grid parameters (lat/lon bounds, spacing)

In [ ]:
print("Running orthorectifier...")
print(f"  Interpolation: {INTERPOLATION}")
print(f"  DEM-corrected: {geo_chip.elevation is not None}")

result = orthorectify(
    geolocation=geo_chip,
    source_array=magnitude,
    metadata=meta,
    interpolation=INTERPOLATION,
    nodata=np.nan,
)

grid = result.output_grid
ortho_magnitude = result.data

print(f"\nOutput grid:")
print(f"  Dimensions  : {grid.rows} x {grid.cols}")
print(f"  Lat range   : [{grid.min_lat:.6f}°, {grid.max_lat:.6f}°]")
print(f"  Lon range   : [{grid.min_lon:.6f}°, {grid.max_lon:.6f}°]")
print(f"  Lat spacing : {grid.lat_spacing:.8f}° ({grid.lat_spacing * 111000:.2f} m)")
print(f"  Lon spacing : {grid.lon_spacing:.8f}° ({grid.lon_spacing * 111000 * np.cos(np.radians(scp.lat)):.2f} m)")

print(f"\nOrthorectified magnitude:")
print(f"  Shape: {ortho_magnitude.shape}, dtype: {ortho_magnitude.dtype}")
print(f"  Valid pixels: {np.isfinite(ortho_magnitude).sum()} / {ortho_magnitude.size}")
print(f"  Range: [{np.nanmin(ortho_magnitude):.2e}, {np.nanmax(ortho_magnitude):.2e}]")

## 4. Visualization with napari

Display **both geometries side-by-side** for direct comparison:
1. **Layer 1:** Slant-range magnitude (native SAR geometry)
2. **Layer 2:** Orthorectified magnitude (WGS-84 geographic grid)

**What to observe:**
- Slant-range image is **vertically compressed** (shorter in azimuth)
- Orthorectified image is **geometrically correct** (taller, proper aspect ratio)
- For graze ~45°, the ortho image should be visibly **stretched** compared to slant range

**napari controls:**
- Toggle layer visibility to switch between geometries
- Use contrast limits slider to adjust dynamic range
- Zoom/pan to inspect local features

In [ ]:
import napari

viewer = napari.Viewer(title="SICD Orthorectification Demo")

# Compute shared percentile limits for consistent contrast
def percentile_limits(img, plow=2.0, phigh=98.0):
    finite = img[np.isfinite(img)]
    return (float(np.percentile(finite, plow)),
            float(np.percentile(finite, phigh)))

vmin, vmax = percentile_limits(magnitude)

# Layer 1: Slant-range magnitude
viewer.add_image(
    magnitude,
    name=f"Slant Range ({chip_rows}×{chip_cols} px)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
)

# Layer 2: Orthorectified magnitude (flip vertically for correct orientation)
viewer.add_image(
    np.flipud(ortho_magnitude),
    name=f"Orthorectified (WGS-84, graze {graze:.1f}°)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
    visible=False,  # start with slant range visible
)

print(f"napari viewer opened with 2 layers")
print(f"Toggle layer visibility to compare slant range vs orthorectified geometry")
print(f"\nExpected visual difference for graze {graze:.1f}°:")
print(f"  - Slant range: vertically compressed (shorter)")
print(f"  - Orthorectified: geometrically correct (taller, proper aspect ratio)")

## 5. Quantitative Comparison

Measure the **geometric distortion** introduced by slant-range projection:

$$\text{Aspect ratio} = \frac{\text{rows}}{\text{cols}}$$

For a scene with graze angle $\theta$, the slant-range aspect ratio is compressed by approximately:

$$\text{compression} \approx \sin(\theta)$$

For $\theta = 45°$, $\sin(45°) \approx 0.707$ → slant range is ~70% as tall as ground range.

In [ ]:
# Aspect ratio comparison
aspect_slant = chip_rows / chip_cols
aspect_ortho = grid.rows / grid.cols
aspect_change = aspect_ortho / aspect_slant

print(f"Slant-range aspect ratio     : {aspect_slant:.4f} (rows/cols)")
print(f"Orthorectified aspect ratio  : {aspect_ortho:.4f} (rows/cols)")
print(f"Aspect ratio change          : {aspect_change:.4f}x (ortho / slant)")
print(f"\nTheoretical compression (sin graze): {np.sin(np.radians(graze)):.4f}")
print(f"Observed compression (slant/ortho) : {aspect_slant / aspect_ortho:.4f}")
print(f"\nInterpretation: Orthorectified image is {aspect_change:.2f}x taller than slant range.")

## Summary

**GRDL orthorectification workflow:**
1. ✅ `SICDGeolocation.from_reader()` — construct geolocation from metadata
2. ✅ `ChipGeolocation()` — wrap for chipped regions (row/col offset)
3. ✅ `open_elevation()` — attach DEM for terrain correction
4. ✅ Convert complex → magnitude **before** resampling (avoid phase cancellation)
5. ✅ `orthorectify()` — auto-grid, DEM-corrected, multi-threaded

**Key insight:** SAR slant-range geometry is **not geographically correct**. Orthorectification is required for:
- Overlay with maps and vector data
- Accurate distance/area measurements
- Multi-temporal change detection
- Terrain analysis

**Performance note:** For the {CHIP_SIZE}×{CHIP_SIZE} chip, orthorectification with DEM lookup and bilinear interpolation should complete in < 10 seconds on a modern CPU.